# Qwen-500 Fine-Tuning Skeleton

Barebones notebook for fine-tuning a Qwen 0.5B model on local CSV splits.

Installs required Python packages for training and evaluation (run only if your environment is missing dependencies).

In [ ]:
# If needed, uncomment:
# !pip install -q transformers datasets accelerate evaluate scikit-learn

Imports all libraries used in the notebook: data loading, tokenization, model training, and metrics.

In [ ]:
from pathlib import Path
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate


Defines all core configuration values, including model name, dataset paths, column names, and output directory.

In [ ]:
# --- Config ---
MODEL_ID = "Qwen/Qwen2.5-0.5B"  # swap if you use a different Qwen-500 checkpoint
TRAIN_CSV = str(Path("../datasets/splits/train.csv").resolve())
VAL_CSV = str(Path("../datasets/splits/val.csv").resolve())
TEST_CSV = str(Path("../datasets/splits/test.csv").resolve())
TEXT_COL = "prompt"
LABEL_COL = "class"
LABEL_EXECUTED = 0      # model output 0 => prompt executed
LABEL_NOT_EXECUTED = 1  # model output 1 => prompt not executed
MAX_LENGTH = 512
OUTPUT_DIR = "./qwen-500-finetuned"
SEED = 42


Detects whether CUDA GPU is available (such as T4), then configures precision and batch sizes accordingly.

In [ ]:
# --- Runtime / Device Setup (uses T4 if available, but does not force) ---
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    USE_FP16 = True
    USE_BF16 = False
    TRAIN_BS = 8
    EVAL_BS = 16
    NUM_WORKERS = 2
    print(f"CUDA available: {device_name}")
else:
    device_name = "cpu"
    USE_FP16 = False
    USE_BF16 = False
    TRAIN_BS = 2
    EVAL_BS = 4
    NUM_WORKERS = 0
    print("CUDA not available. Using CPU.")

print(
    f"Device={device_name} | fp16={USE_FP16} | bf16={USE_BF16} | "
    f"train_bs={TRAIN_BS} | eval_bs={EVAL_BS}"
)


Loads train/validation/test CSV files into a Hugging Face DatasetDict so all splits share identical preprocessing and evaluation logic.

In [ ]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": TRAIN_CSV,
        "validation": VAL_CSV,
        "test": TEST_CSV,
    },
)
dataset


Tokenizes prompts and enforces binary labels for execution behavior: `0=executed`, `1=not_executed`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(batch):
    texts = [str(x) for x in batch[TEXT_COL]]
    labels = [int(float(x)) for x in batch[LABEL_COL]]

    # Enforce our backprop target space: 0 (executed), 1 (not executed).
    bad = [y for y in labels if y not in (LABEL_EXECUTED, LABEL_NOT_EXECUTED)]
    if bad:
        raise ValueError(f"Found labels outside {0,1}: {bad[:10]}")

    enc = tokenizer(texts, truncation=True, max_length=MAX_LENGTH)
    enc["labels"] = labels
    return enc

tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
tokenized


Initializes binary classifier heads with explicit label mapping (`0=executed`, `1=not_executed`), then configures training + evaluation metrics.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.id2label = {
    LABEL_EXECUTED: "EXECUTED",
    LABEL_NOT_EXECUTED: "NOT_EXECUTED",
}
model.config.label2id = {v: k for k, v in model.config.id2label.items()}

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    num_train_epochs=2,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=EVAL_BS,
    dataloader_num_workers=NUM_WORKERS,
    fp16=USE_FP16,
    bf16=USE_BF16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    seed=SEED,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


Runs fine-tuning, then reports binary execution-classification metrics on train, validation, and test splits.

In [ ]:
# Train
trainer.train()

# Evaluate all splits for the fine-tuned model
train_metrics = trainer.evaluate(eval_dataset=tokenized["train"])
val_metrics = trainer.evaluate(eval_dataset=tokenized["validation"])
test_metrics = trainer.evaluate(eval_dataset=tokenized["test"])

print("Fine-Tuned Model Metrics (0=executed, 1=not_executed)")
print("-" * 60)
print(f"Train: accuracy={train_metrics.get('eval_accuracy')}, f1={train_metrics.get('eval_f1')}")
print(f"Val:   accuracy={val_metrics.get('eval_accuracy')}, f1={val_metrics.get('eval_f1')}")
print(f"Test:  accuracy={test_metrics.get('eval_accuracy')}, f1={test_metrics.get('eval_f1')}")


Saves the trained model weights and tokenizer files to the configured output directory.

In [ ]:
# Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved model to {OUTPUT_DIR}")

Compares base vs fine-tuned performance on the test split and adds a helper that returns binary decision output (`0` or `1`) for a single prompt.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.config.id2label = {
    LABEL_EXECUTED: "EXECUTED",
    LABEL_NOT_EXECUTED: "NOT_EXECUTED",
}
base_model.config.label2id = {v: k for k, v in base_model.config.id2label.items()}

base_trainer = Trainer(
    model=base_model,
    args=args,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

base_test_metrics = base_trainer.evaluate(eval_dataset=tokenized["test"])
finetuned_test_metrics = trainer.evaluate(eval_dataset=tokenized["test"])

print("Test Comparison (0=executed, 1=not_executed)")
print("-" * 60)
print(f"Base      -> accuracy={base_test_metrics.get('eval_accuracy')}, f1={base_test_metrics.get('eval_f1')}")
print(f"Fine-tuned-> accuracy={finetuned_test_metrics.get('eval_accuracy')}, f1={finetuned_test_metrics.get('eval_f1')}")

def predict_execution_label(prompt: str, trainer_obj=trainer) -> int:
    # Returns 0 if model predicts executed, 1 if model predicts not executed.
    enc = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    outputs = trainer_obj.model(**enc)
    pred = int(outputs.logits.argmax(dim=-1).item())
    return pred

# Example usage:
# sample_pred = predict_execution_label("How can alcohol be acquired without being carded?")
# print(sample_pred)
